# CSL7110 — Assignment 4: Clustering and PageRank

**Parts:** K-Center / K-Means++, Inverted Index & Web Search, PageRank on Spark  
**Name:** Kakarla Sai Swaroop
**Roll No.:** M25DE1023  
**Mail ID:** m25de1023@iitj.ac.in

---
> **Environment note:** The algorithms implement the Spark programming model semantics.  
> Core logic runs in pure Python/NumPy for portability.  
> Full PySpark code is provided in Part 3's final cell

---
## Part 1 — Clustering (40 Marks)

### Algorithms

**Farthest-First Traversal (K-Center):**  
Greedily picks k centers by always selecting the point farthest from any already-chosen center.  
Guarantees a **2-approximation** to optimal k-center. Time: O(|P|·k).

**K-Means++ Seeding:**  
Chooses centers probabilistically: first center is uniform random; subsequent centers are sampled with probability **proportional to D²** (squared distance to nearest existing center).  
Provides **O(log k)** approximation and far better real-world results than random initialization. Time: O(|P|·k).

**K-Means Objective:**  
`kmeansObj(P, C) = (1/|P|) · Σ_{p∈P} min_{c∈C} ||p−c||²`

In [30]:
# ============================================================
# COLAB SETUP
# ============================================================
!rm -rf "/content/Assignment 4- datasets"
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Extract dataset zip from Drive
import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/Assignment_4-_datasets.zip'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

print("Dataset extracted")

# 3. Set correct BASE path
BASE = "/content/Assignment 4- datasets"

print("Folders inside BASE:")
print(os.listdir(BASE))

# 4. Rename folders to avoid space based issues
os.rename(f"{BASE}/Q1- UCI Spam clustering", f"{BASE}/Q1_UCI_Spam")
os.rename(f"{BASE}/Q2- webSearch", f"{BASE}/Q2_webSearch")

print("Folders renamed")
print("After rename:", os.listdir(BASE))

# 5. Download PageRank graphs
os.makedirs('/content/graph', exist_ok=True)

!wget -q -O /content/graph/small.txt \
  "https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/refs/heads/main/graphs/small.txt"

!wget -q -O /content/graph/whole.txt \
  "https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/refs/heads/main/graphs/whole.txt"

print("Graph files downloaded")

# 6. Set paths (FIXED)
SPAM_PATH    = f"{BASE}/Q1_UCI_Spam/spambase.data"
WEBPAGES_DIR = f"{BASE}/Q2_webSearch/webpages/"
ACTIONS_FILE = f"{BASE}/Q2_webSearch/actions.txt"
ANSWERS_FILE = f"{BASE}/Q2_webSearch/answers.txt"
SMALL_PATH   = "/content/graph/small.txt"
WHOLE_PATH   = "/content/graph/whole.txt"

print("Paths set")

# 7. Install PySpark
!pip install pyspark -q
print("PySpark installed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted
Folders inside BASE:
['Q2- webSearch', 'Q1- UCI Spam clustering']
Folders renamed
After rename: ['Q1_UCI_Spam', 'Q2_webSearch']
Graph files downloaded
Paths set
PySpark installed


In [31]:
import os

BASE = "/content/Assignment 4- datasets"

paths_to_check = [
    f"{BASE}/Q1_UCI_Spam/spambase.data",
    f"{BASE}/Q2_webSearch/actions.txt",
    f"{BASE}/Q2_webSearch/answers.txt",
    f"{BASE}/Q2_webSearch/webpages/",
]

for p in paths_to_check:
    status = "Found" if os.path.exists(p) else "NOT FOUND"
    print(f"{status}  →  {p}")

Found  →  /content/Assignment 4- datasets/Q1_UCI_Spam/spambase.data
Found  →  /content/Assignment 4- datasets/Q2_webSearch/actions.txt
Found  →  /content/Assignment 4- datasets/Q2_webSearch/answers.txt
Found  →  /content/Assignment 4- datasets/Q2_webSearch/webpages/


In [3]:
# ============================================================
# PART 1 — IMPORTS
# ============================================================
import numpy as np
import random
import time

# Reproducibility
random.seed(42)
np.random.seed(42)

print("Imports OK.")

Imports OK.


In [4]:
# ============================================================
# FUNCTION 1: readVectorsSeq
# Reads a comma-separated feature file; drops the last column (label).
# Returns a Python list of numpy arrays.
#
# Spark equivalent:
#   sc.textFile(filename)
#     .map(lambda line: Vectors.dense(
#           [float(x) for x in line.split(',')[:-1]]))
#     .collect()
# ============================================================
def readVectorsSeq(filename):
    """
    Read a CSV file with one feature vector per line.
    The last column (class label) is dropped.
    Returns a list of np.ndarray (one per row).
    """
    points = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = list(map(float, line.split(',')))
            points.append(np.array(vals[:-1]))   # drop label column
    return points


# Helper: squared Euclidean distance  — equiv. to Vectors.sqdist(a, b)
def sq_dist(a, b):
    diff = a - b
    return float(np.dot(diff, diff))


print("readVectorsSeq and sq_dist defined.")

readVectorsSeq and sq_dist defined.


In [5]:
# ============================================================
# FUNCTION 2: kcenter  (Farthest-First Traversal)
# Time: O(|P| * k)  — one linear scan per new center.
# ============================================================
def kcenter(P, k):
    """
    K-Center clustering by Farthest-First Traversal.

    Algorithm:
      1. Pick P[0] as the first center.
      2. Repeat k-1 times:
           Pick the point that is FARTHEST from the current set of centers.

    Spark equivalent:
      centers = [P.first()]
      bc = sc.broadcast(centers)
      for _ in range(k - 1):
          farthest = (
              P.map(lambda p: (p, min(Vectors.sqdist(p, c) for c in bc.value)))
               .reduce(lambda a, b: a if a[1] > b[1] else b)[0]
          )
          centers.append(farthest)
          bc = sc.broadcast(centers)

    Returns: list of k numpy arrays (centers).
    """
    centers = [P[0]]
    for _ in range(k - 1):
        farthest = max(P, key=lambda p: min(sq_dist(p, c) for c in centers))
        centers.append(farthest)
    return centers


print("kcenter defined.")

kcenter defined.


In [6]:
# ============================================================
# FUNCTION 3: kmeansPP  (K-Means++ seeding)
# Time: O(|P| * k) — one linear scan per new center.
# ============================================================
def kmeansPP(P, k):
    """
    K-Means++ center initialization using D² sampling.

    Algorithm:
      1. Pick a uniformly random point as the first center.
      2. Repeat k-1 times:
           a. Compute D²[p] = min distance squared from p to nearest center.
           b. Sample next center with probability proportional to D²[p].

    This ensures centers are spread out, giving O(log k) approximation.

    Spark equivalent:
      centers = [P.takeSample(False, 1)[0]]
      bc = sc.broadcast(centers)
      for _ in range(k - 1):
          dists_rdd = P.map(
              lambda p: (p, min(Vectors.sqdist(p, c) for c in bc.value))
          ).cache()
          total = dists_rdd.map(lambda x: x[1]).sum()
          # weighted sampling from the RDD using custom partitioned approach
          ...

    Returns: list of k numpy arrays (centers).
    """
    centers = [P[random.randint(0, len(P) - 1)]]   # Step 1: random first center
    for _ in range(k - 1):
        # Step 2a: D² distances
        dists = np.array([min(sq_dist(p, c) for c in centers) for p in P])
        # Step 2b: Weighted random sample
        probs = dists / dists.sum()
        idx = np.random.choice(len(P), p=probs)
        centers.append(P[idx])
    return centers


print("kmeansPP defined.")

kmeansPP defined.


In [7]:
# ============================================================
# FUNCTION 4: kmeansObj  (K-Means Objective / Cost Function)
# ============================================================
def kmeansObj(P, C):
    """
    Compute average squared distance from each point to its nearest center.
    kmeansObj = (1/|P|) * sum_{p in P} min_{c in C} ||p - c||^2

    Lower = better (tighter clustering).

    Spark equivalent:
      bc = sc.broadcast(C)
      P.map(lambda p: min(Vectors.sqdist(p, c) for c in bc.value)).mean()
    """
    total = sum(min(sq_dist(p, c) for c in C) for p in P)
    return total / len(P)


print("kmeansObj defined.")

kmeansObj defined.


In [8]:
# ============================================================
# LOAD DATASET
# ============================================================

BASE = "/content/Assignment 4- datasets"

SPAM_PATH = f"{BASE}/Q1_UCI_Spam/spambase.data"

P = readVectorsSeq(SPAM_PATH)

print(f"Dataset: {len(P)} points, {len(P[0])} dimensions")
print(f"First point (first 5 features): {P[0][:5]}")

# Parameters
k  = 3
k1 = 10

print(f"Parameters: k={k}, k1={k1}")

Dataset: 4601 points, 57 dimensions
First point (first 5 features): [0.   0.64 0.64 0.   0.32]
Parameters: k=3, k1=10


In [10]:
# ============================================================
# EXPERIMENT 1 — kcenter(P, k): Farthest-First k-Center
# ============================================================
print("=" * 55)
print("EXPERIMENT 1: kcenter(P, k)")
print("=" * 55)

t0 = time.time()
C_kc = kcenter(P, k)
elapsed = time.time() - t0

print(f"Running time: {elapsed:.4f} seconds")
print(f"Centers returned: {len(C_kc)}")

# Compute objective for reference
obj_kc = kmeansObj(P, C_kc)
print(f"kmeansObj (for reference): {obj_kc:.4f}")

EXPERIMENT 1: kcenter(P, k)
Running time: 0.0407 seconds
Centers returned: 3
kmeansObj (for reference): 290451.5724


In [11]:
# ============================================================
# EXPERIMENT 2 — kmeansPP(P, k) then kmeansObj(P, C)
# ============================================================
print("=" * 55)
print("EXPERIMENT 2: kmeansPP(P, k) -> kmeansObj(P, C)")
print("=" * 55)

t0 = time.time()
C_kpp = kmeansPP(P, k)
elapsed_kpp = time.time() - t0

obj_kpp = kmeansObj(P, C_kpp)

print(f"kmeansPP running time: {elapsed_kpp:.4f} seconds")
print(f"kmeansObj(P, C) = {obj_kpp:.4f}")

EXPERIMENT 2: kmeansPP(P, k) -> kmeansObj(P, C)
kmeansPP running time: 0.0446 seconds
kmeansObj(P, C) = 151995.4489


In [12]:
# ============================================================
# EXPERIMENT 3 — Coreset: kcenter(P, k1) -> kmeansPP(X, k) -> kmeansObj(P, C)
#
# Idea: kcenter builds a 'coreset' X of k1 well-spread representatives.
# kmeans++ on this small set X is fast and produces good seeds.
# We then evaluate the final k centers on the original full set P.
# Larger k1 → richer coreset → better final centers.
# ============================================================
print("=" * 55)
print("EXPERIMENT 3: kcenter(P, k1) -> kmeansPP(X, k) -> kmeansObj(P, C)")
print("=" * 55)

# Stage A: Build coreset
t0 = time.time()
X = kcenter(P, k1)
t_kc = time.time() - t0
print(f"Stage A — kcenter(P, k1={k1}) time: {t_kc:.4f}s | Coreset size: {len(X)}")

# Stage B: kmeans++ on coreset
t0 = time.time()
C_combo = kmeansPP(X, k)
t_kpp = time.time() - t0
print(f"Stage B — kmeansPP(X, k={k}) time:  {t_kpp:.4f}s")

# Stage C: Evaluate on full P
obj_combo = kmeansObj(P, C_combo)
print(f"kmeansObj(P, C) = {obj_combo:.4f}")

print()
print("-" * 55)
print("SUMMARY")
print("-" * 55)
print(f"  Exp 2 — kmeans++ directly on P (k={k}):          {obj_kpp:.2f}")
print(f"  Exp 3 — coreset(k1={k1}) -> kmeans++(k={k}):     {obj_combo:.2f}")
print()
print("Observation:")
print("  Both experiments produce k=3 centers for the spam dataset.")
print("  Experiment 3 first compresses P down to k1=10 coreset points")
print("  using kcenter, then runs kmeans++ on that small set. Because")
print("  the coreset is tiny (10 points), the seeding is fast but coarser.")
print("  Increasing k1 (e.g., 50, 100) will generally improve Exp 3's result.")

EXPERIMENT 3: kcenter(P, k1) -> kmeansPP(X, k) -> kmeansObj(P, C)
Stage A — kcenter(P, k1=10) time: 0.4812s | Coreset size: 10
Stage B — kmeansPP(X, k=3) time:  0.0009s
kmeansObj(P, C) = 264462.8954

-------------------------------------------------------
SUMMARY
-------------------------------------------------------
  Exp 2 — kmeans++ directly on P (k=3):          151995.45
  Exp 3 — coreset(k1=10) -> kmeans++(k=3):     264462.90

Observation:
  Both experiments produce k=3 centers for the spam dataset.
  Experiment 3 first compresses P down to k1=10 coreset points
  using kcenter, then runs kmeans++ on that small set. Because
  the coreset is tiny (10 points), the seeding is fast but coarser.
  Increasing k1 (e.g., 50, 100) will generally improve Exp 3's result.


---
## Part 2 — Web Search: Inverted Index (40 Marks)

### Design Overview

| Class | Role |
|---|---|
| `Position` | `<PageEntry, word_index>` tuple |
| `WordEntry` | All positions across all pages for one word |
| `PageIndex` | Per-page map: word → WordEntry |
| `PageEntry` | Reads a file, builds its PageIndex |
| `MyHashTable` | Custom hash table (polynomial hash) |
| `InvertedPageIndex` | Global aggregator across all pages |
| `SearchEngine` | Processes action commands |

### Text Processing Rules
- All words **lowercased**.
- **Stop words** skipped in index but **counted** in position numbering.
- **Punctuation** replaced with a space.
- **Singular/plural** normalization: `stacks→stack`, `structures→structure`, `applications→application`.
- Query words with non-alpha characters (e.g. `C++`) have non-alpha chars stripped before lookup.

In [13]:
# ============================================================
# PART 2 — CONSTANTS
# ============================================================
import os, re

# Connector words to skip during indexing (exhaustive list from assignment)
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

# Punctuation to replace with space (exhaustive list from assignment)
PUNCT_CHARS = set('{}[]<>=().,;\'"?#!-:')

# Singular→singular normalization (exhaustive list from assignment)
SINGULAR_MAP = {
    'stacks':       'stack',
    'structures':   'structure',
    'applications': 'application',
}

BASE = "/content/Assignment 4- datasets"
WEBPAGES_DIR = f"{BASE}/Q2_webSearch/webpages/"

print("Constants OK.")

Constants OK.


In [14]:
# ============================================================
# TEXT UTILITIES
# ============================================================

def replace_punctuation(text):
    """Replace each character in PUNCT_CHARS with a space."""
    for ch in PUNCT_CHARS:
        text = text.replace(ch, ' ')
    return text


def normalize_word(word):
    """Lowercase and apply singular-plural map."""
    return SINGULAR_MAP.get(word.lower(), word.lower())


def normalize_query_word(word):
    """
    For query words: strip non-alpha characters first, then normalize.
    This handles tokens like 'C++' (stored as 'c' in the index).
    """
    cleaned = re.sub(r'[^a-zA-Z]', '', word)
    return normalize_word(cleaned)


def tokenize(text):
    """Replace punctuation and split into lowercase tokens."""
    return replace_punctuation(text).lower().split()


print("Text utilities OK.")

Text utilities OK.


In [15]:
# ============================================================
# CLASS: Position — <page, word_index> tuple
# ============================================================
class Position:
    def __init__(self, page_entry, word_index):
        """
        Constructor.
        :param page_entry:  The PageEntry this position belongs to.
        :param word_index:  1-based position in the document (all tokens counted).
        """
        self._page = page_entry
        self._idx  = word_index

    def getPageEntry(self):
        """Return the associated PageEntry."""
        return self._page

    def getWordIndex(self):
        """Return 1-based word position."""
        return self._idx

    def __repr__(self):
        return f"Position({self._page.page_name}, {self._idx})"


print("Position OK.")

Position OK.


In [16]:
# ============================================================
# CLASS: WordEntry — positions for a single word across document(s)
# ============================================================
class WordEntry:
    def __init__(self, word):
        self.word      = word
        self._positions = []

    def addPosition(self, position):
        """Add a single Position."""
        self._positions.append(position)

    def addPositions(self, positions):
        """Add a list of Position objects."""
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        """Return all stored Position objects."""
        return self._positions

    def getTermFrequency(self, page_entry):
        """
        tf = (occurrences of word in page) / (total tokens in page).
        """
        count = sum(1 for p in self._positions if p.getPageEntry() is page_entry)
        return count / page_entry.total_words if page_entry.total_words else 0.0

    def __repr__(self):
        return f"WordEntry('{self.word}', {len(self._positions)} positions)"


print("WordEntry OK.")

WordEntry OK.


In [17]:
# ============================================================
# CLASS: PageIndex — per-page word map
# ============================================================
class PageIndex:
    def __init__(self):
        self._entries = {}   # normalized_word -> WordEntry

    def addPositionForWord(self, word_str, position):
        """
        Add position to the WordEntry for word_str.
        Creates a new WordEntry if the word hasn't been seen yet.
        """
        if word_str not in self._entries:
            self._entries[word_str] = WordEntry(word_str)
        self._entries[word_str].addPosition(position)

    def getWordEntries(self):
        """Return all WordEntry objects for this page."""
        return list(self._entries.values())

    def getWordEntry(self, word_str):
        """Return WordEntry for word_str, or None."""
        return self._entries.get(word_str)


print("PageIndex OK.")

PageIndex OK.


In [18]:
# ============================================================
# CLASS: PageEntry — reads a webpage file and indexes it
# ============================================================
class PageEntry:
    def __init__(self, page_name, webpages_dir):
        """
        Constructor. Reads the file and builds the per-page index.
        :param page_name:    Filename (no path) of the webpage.
        :param webpages_dir: Directory containing webpage files.
        """
        self.page_name   = page_name
        self._page_index = PageIndex()
        self.total_words = 0
        self._build_index(os.path.join(webpages_dir, page_name))

    def _build_index(self, filepath):
        with open(filepath, 'r', errors='ignore') as f:
            text = f.read()

        tokens    = tokenize(text)
        word_pos  = 0   # 1-based counter for ALL tokens

        for token in tokens:
            word_pos += 1                          # count every token for position

            if token in STOP_WORDS:                # skip stop words (still counted)
                continue
            if not token.isalpha():                # skip non-alpha (numbers, C++, etc.)
                continue

            norm = normalize_word(token)           # lowercase + singular map
            self._page_index.addPositionForWord(norm, Position(self, word_pos))

        self.total_words = word_pos

    def getPageIndex(self):
        return self._page_index

    def __repr__(self):
        return self.page_name


print("PageEntry OK.")

PageEntry OK.


In [19]:
# ============================================================
# CLASS: MyHashTable — custom hash table for InvertedPageIndex
# Polynomial rolling hash; each bucket stores {word: WordEntry}.
# ============================================================
class MyHashTable:
    def __init__(self, size=2048):
        self._size  = size
        self._table = [None] * size

    def getHashIndex(self, word_str):
        """Polynomial rolling hash: h = Σ ord(c) * 31^i  mod size."""
        h = 0
        for ch in word_str:
            h = (h * 31 + ord(ch)) % self._size
        return h

    def addPositionsForWord(self, word_entry):
        """
        Insert or merge a WordEntry into the table.
        If an entry for this word already exists, append positions to it.
        """
        idx = self.getHashIndex(word_entry.word)
        if self._table[idx] is None:
            self._table[idx] = {}
        if word_entry.word not in self._table[idx]:
            self._table[idx][word_entry.word] = WordEntry(word_entry.word)
        self._table[idx][word_entry.word].addPositions(
            word_entry.getAllPositionsForThisWord()
        )

    def getWordEntry(self, word_str):
        """Return the WordEntry for word_str, or None if absent."""
        idx    = self.getHashIndex(word_str)
        bucket = self._table[idx]
        return bucket.get(word_str) if bucket else None


print("MyHashTable OK.")

MyHashTable OK.


In [20]:
# ============================================================
# CLASS: InvertedPageIndex — global aggregator
# ============================================================
class InvertedPageIndex:
    def __init__(self):
        self._hashtable = MyHashTable()
        self._pages     = []

    def addPage(self, page_entry):
        """Add a page and merge all its word entries into the global table."""
        self._pages.append(page_entry)
        for we in page_entry.getPageIndex().getWordEntries():
            self._hashtable.addPositionsForWord(we)

    def getPagesWhichContainWord(self, word_str):
        """Return set of PageEntry objects containing word_str (normalized)."""
        we = self._hashtable.getWordEntry(word_str)
        if we is None:
            return set()
        return {pos.getPageEntry() for pos in we.getAllPositionsForThisWord()}

    def getPages(self):
        return self._pages


print("InvertedPageIndex OK.")

InvertedPageIndex OK.


In [21]:
# ============================================================
# CLASS: SearchEngine — processes action strings
# ============================================================
class SearchEngine:
    def __init__(self, webpages_dir):
        self._index       = InvertedPageIndex()
        self._webpages_dir = webpages_dir

    def performAction(self, action_message):
        """
        Parse and execute one action. Returns output string or None.

        Supported actions:
          addPage <name>
          queryFindPagesWhichContainWord <word>
          queryFindPositionsOfWordInAPage <word> <page>
        """
        parts = action_message.strip().split()
        if not parts:
            return None
        cmd = parts[0]

        # ---- addPage ----
        if cmd == 'addPage':
            pe = PageEntry(parts[1], self._webpages_dir)
            self._index.addPage(pe)
            return None

        # ---- queryFindPagesWhichContainWord ----
        elif cmd == 'queryFindPagesWhichContainWord':
            raw  = parts[1]
            norm = normalize_query_word(raw)   # strip non-alpha, lowercase, singular map
            pages = self._index.getPagesWhichContainWord(norm)
            if not pages:
                return f"No webpage contains word {raw}"
            return ', '.join(sorted(p.page_name for p in pages))

        # ---- queryFindPositionsOfWordInAPage ----
        elif cmd == 'queryFindPositionsOfWordInAPage':
            raw_word, page_name = parts[1], parts[2]
            norm_word = normalize_query_word(raw_word)

            # Check page is in the database
            pe = next((p for p in self._index.getPages() if p.page_name == page_name), None)
            if pe is None:
                return f"No webpage {page_name} found"

            we = self._index._hashtable.getWordEntry(norm_word)
            if we is None:
                return f"Webpage {page_name} does not contain word {raw_word}"

            pos_list = [p.getWordIndex()
                        for p in we.getAllPositionsForThisWord()
                        if p.getPageEntry() is pe]
            if not pos_list:
                return f"Webpage {page_name} does not contain word {raw_word}"

            return ', '.join(str(i) for i in sorted(pos_list))

        return f"Unknown command: {cmd}"


print("SearchEngine OK.")

SearchEngine OK.


In [22]:
# ============================================================
# RUN actions.txt and verify against answers.txt
# ============================================================


BASE = "/content/Assignment 4- datasets"

ACTIONS_FILE = f"{BASE}/Q2_webSearch/actions.txt"
ANSWERS_FILE = f"{BASE}/Q2_webSearch/answers.txt"

engine = SearchEngine(WEBPAGES_DIR)

with open(ACTIONS_FILE, 'r') as f:
    actions = [ln.strip() for ln in f if ln.strip()]

with open(ANSWERS_FILE, 'r') as f:
    expected = [ln.strip() for ln in f if ln.strip()]

outputs = []
for action in actions:
    r = engine.performAction(action)
    if r is not None:
        outputs.append(r)

print(f"{'OUTPUT':<60} {'EXPECTED':<60} OK?")
print("-" * 125)
all_ok = True
for got, exp in zip(outputs, expected):
    ok = got == exp
    if not ok:
        all_ok = False
    print(f"{got:<60} {exp:<60} {'✓' if ok else '✗ MISMATCH'}")

print()
print("ALL OUTPUTS MATCH!" if all_ok else "Some outputs differ — check above.")

OUTPUT                                                       EXPECTED                                                     OK?
-----------------------------------------------------------------------------------------------------------------------------
No webpage contains word delhi                               No webpage contains word delhi                               ✓
stack_datastructure_wiki                                     stack_datastructure_wiki                                     ✓
stack_datastructure_wiki                                     stack_datastructure_wiki                                     ✓
Webpage stack_datastructure_wiki does not contain word magazines Webpage stack_datastructure_wiki does not contain word magazines ✓
No webpage contains word allain                              No webpage contains word allain                              ✓
stack_cprogramming                                           stack_cprogramming                                         

---
## Part 3 — PageRank on Spark (40 Marks)

**Dataset:** [pnijhara/PySpark-PageRank](https://github.com/pnijhara/PySpark-PageRank/tree/main/graph)  
Download `small.txt` (53 nodes) and `whole.txt` (1000 nodes, 8192 edges) into a `graph/` subfolder.

### Algorithm

Initialize $r^{(0)} = \frac{1}{n}\mathbf{1}$, then iterate for 40 steps:

$$r^{(i)} = \frac{1-\beta}{n}\mathbf{1} + \beta M r^{(i-1)}$$

where $M_{j,i} = \frac{1}{\deg(i)}$ if edge $(i \to j)$ exists, else 0, and $\beta = 0.8$.

### Spark Strategy
1. Load edge list → `.distinct()` to deduplicate.
2. `.groupByKey()` → adjacency list RDD (cached).
3. Each iteration: `links.join(ranks).flatMap(contributions)` → `.reduceByKey(sum)` → add teleportation.

In [23]:
# ============================================================
# PART 3 — PageRank Implementation
# ============================================================
import os
import time
from collections import defaultdict


"""def load_edges(filepath):

    Load edges from a whitespace-separated edge list.
    Returns a set of (src, dst) tuples (deduplicated, no self-loops).

    Spark equivalent:
      sc.textFile(filepath)
        .map(lambda l: tuple(map(int, l.split())))
        .filter(lambda e: e[0] != e[1])
        .distinct()

    edge_set = set()
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            src, dst = int(parts[0]), int(parts[1])
            if src != dst:
                edge_set.add((src, dst))
    return edge_set"""
def load_edges(filepath):
    edges = set()

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            # 🔥 IMPORTANT FIX
            if len(parts) != 2:
                continue

            try:
                src, dst = int(parts[0]), int(parts[1])
            except:
                continue

            if src != dst:
                edges.add((src, dst))

    return edges

def pagerank(edges, n, beta=0.8, iters=40, verbose=False):
    """
    Iterative PageRank.

    :param edges:  set of (src, dst) tuples (deduped, no self-loops)
    :param n:      number of nodes
    :param beta:   damping factor
    :param iters:  number of iterations
    :param verbose: print progress
    :return: dict {node_id: score}

    ===== EQUIVALENT SPARK CODE (paste into PySpark) =====
    # Build adjacency RDD
    links = (
        edges.groupByKey()
             .mapValues(list)
             .cache()
    )
    ranks = links.mapValues(lambda _: 1.0 / n)

    for i in range(iters):
        contribs = links.join(ranks).flatMap(
            lambda kv: [(dst, beta * kv[1][1] / len(kv[1][0]))
                        for dst in kv[1][0]]
        )
        ranks = (
            contribs.reduceByKey(lambda a, b: a + b)
                    .mapValues(lambda v: (1 - beta) / n + v)
        )
    return ranks.collect()
    =======================================================
    """
    # Collect all nodes
    nodes = sorted({u for e in edges for u in e})

    # Build adjacency list and out-degrees
    adj     = defaultdict(list)
    out_deg = defaultdict(int)
    for src, dst in edges:
        adj[src].append(dst)
        out_deg[src] += 1

    # Initialize rank vector: r[v] = 1/n
    r = {v: 1.0 / n for v in nodes}
    teleport = (1.0 - beta) / n

    for i in range(iters):
        new_r = {v: teleport for v in nodes}
        # Distribute rank: Spark flatMap + reduceByKey
        for src, neighbors in adj.items():
            contrib = beta * r[src] / out_deg[src]
            for dst in neighbors:
                new_r[dst] += contrib
        r = new_r
        if verbose and (i + 1) % 10 == 0:
            print(f"  Iteration {i+1:2d} done.")

    return r


print("PageRank functions defined.")

PageRank functions defined.


In [24]:
print(os.path.exists("/content/graph/small.txt"))
with open("/content/graph/small.txt") as f:
    for _ in range(5):
        print(f.readline())
edges = load_edges("/content/graph/small.txt")
print("Edges loaded:", len(edges))
print(list(edges)[:5])

with open("/content/graph/small.txt") as f:
    for i in range(10):
        line = f.readline()
        print(repr(line))

True
100	1

13	1

28	1

89	1

82	1

Edges loaded: 950
[(58, 1), (71, 20), (36, 53), (90, 33), (55, 66)]
'100\t1\n'
'13\t1\n'
'28\t1\n'
'89\t1\n'
'82\t1\n'
'30\t1\n'
'79\t1\n'
'65\t1\n'
'88\t1\n'
'25\t1\n'


In [25]:
# ============================================================
# EXPERIMENT A — small.txt validation (expected top score ≈ 0.036)
# ============================================================
SMALL_PATH = "graph/small.txt"
WHOLE_PATH = "graph/whole.txt"

if os.path.exists(SMALL_PATH):
    print("=" * 55)
    print("SMALL GRAPH VALIDATION")
    print("=" * 55)
    small_edges = load_edges(SMALL_PATH)
    all_nodes   = {u for e in small_edges for u in e}
    n_small     = len(all_nodes)
    print(f"Nodes: {n_small}  |  Edges (deduped): {len(small_edges)}")

    r_small = pagerank(small_edges, n=n_small, beta=0.8, iters=40)
    sorted_s = sorted(r_small.items(), key=lambda x: -x[1])

    top_score = sorted_s[0][1]
    print(f"Top score: {top_score:.4f}  (expected ≈ 0.036)")
    print()
    print("Top 5 nodes:")
    for node, score in sorted_s[:5]:
        print(f"  Node {node:4d} — score = {score:.6f}")
    print()
    print("Bottom 5 nodes:")
    for node, score in sorted_s[-5:]:
        print(f"  Node {node:4d} — score = {score:.6f}")
else:
    print(f"[INFO] {SMALL_PATH} not found.")
    print("Download from: https://github.com/pnijhara/PySpark-PageRank/tree/main/graph")

SMALL GRAPH VALIDATION
Nodes: 100  |  Edges (deduped): 950
Top score: 0.0357  (expected ≈ 0.036)

Top 5 nodes:
  Node   53 — score = 0.035731
  Node   14 — score = 0.034171
  Node   40 — score = 0.033630
  Node    1 — score = 0.030006
  Node   27 — score = 0.029720

Bottom 5 nodes:
  Node   89 — score = 0.003922
  Node   37 — score = 0.003808
  Node   81 — score = 0.003695
  Node   59 — score = 0.003670
  Node   85 — score = 0.003410


In [26]:
# ============================================================
# EXPERIMENT B — whole.txt (1000 nodes, ~8192 edges)
#   β = 0.8, 40 iterations
# ============================================================
if os.path.exists(WHOLE_PATH):
    print("=" * 55)
    print("FULL GRAPH — whole.txt")
    print("=" * 55)

    t0 = time.time()
    whole_edges = load_edges(WHOLE_PATH)
    all_nodes_w = {u for e in whole_edges for u in e}
    n_whole     = len(all_nodes_w)
    print(f"Nodes: {n_whole}  |  Edges (deduped): {len(whole_edges)}")

    t1 = time.time()
    r_whole = pagerank(whole_edges, n=n_whole, beta=0.8, iters=40, verbose=True)
    t2 = time.time()
    print(f"Total computation time: {t2 - t1:.3f}s")

    sorted_w = sorted(r_whole.items(), key=lambda x: -x[1])

    print()
    print("Top 5 nodes (highest PageRank):")
    for rank_pos, (node, score) in enumerate(sorted_w[:5], 1):
        print(f"  #{rank_pos}  Node {node:4d}  Score = {score:.6f}")

    print()
    print("Bottom 5 nodes (lowest PageRank):")
    for rank_pos, (node, score) in enumerate(sorted_w[-5:], n_whole - 4):
        print(f"  #{rank_pos}  Node {node:4d}  Score = {score:.6f}")
else:
    print(f"[INFO] {WHOLE_PATH} not found.")
    print("Download from: https://github.com/pnijhara/PySpark-PageRank/tree/main/graph")

FULL GRAPH — whole.txt
Nodes: 1000  |  Edges (deduped): 8161
  Iteration 10 done.
  Iteration 20 done.
  Iteration 30 done.
  Iteration 40 done.
Total computation time: 0.052s

Top 5 nodes (highest PageRank):
  #1  Node  263  Score = 0.002020
  #2  Node  537  Score = 0.001943
  #3  Node  965  Score = 0.001925
  #4  Node  243  Score = 0.001853
  #5  Node  285  Score = 0.001827

Bottom 5 nodes (lowest PageRank):
  #996  Node  408  Score = 0.000388
  #997  Node  424  Score = 0.000355
  #998  Node   62  Score = 0.000353
  #999  Node   93  Score = 0.000351
  #1000  Node  558  Score = 0.000329


In [29]:
# ============================================================
# PART 3 — PageRank on PySpark (actually runs it)
# ============================================================
!pip install pyspark -q

from pyspark import SparkContext
import time

# Stop any existing Spark context first
try:
    sc.stop()
except:
    pass

sc = SparkContext("local[*]", "PageRank")
sc.setLogLevel("WARN")

BETA  = 0.8
ITERS = 40

# ── SMALL GRAPH FIRST (validation) ──────────────────────────
!wget -q -O /content/small.txt \
  "https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/refs/heads/main/graphs/small.txt"
!wget -q -O /content/whole.txt \
  "https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/refs/heads/main/graphs/whole.txt"

# ── Run on small.txt first ───────────────────────────────────
print("=" * 50)
print("SMALL GRAPH (53 nodes) — expected top score ≈ 0.036")
print("=" * 50)

raw_small = sc.textFile("/content/small.txt")
edges_small = (
    raw_small
    .map(lambda line: tuple(map(int, line.strip().split())))
    .filter(lambda e: e[0] != e[1])
    .distinct()
)

N_small = edges_small.flatMap(lambda e: [e[0], e[1]]).distinct().count()
print(f"Nodes: {N_small}")

links_small = edges_small.groupByKey().mapValues(list).cache()
ranks_small = links_small.mapValues(lambda _: 1.0 / N_small)

for i in range(ITERS):
    contribs = links_small.join(ranks_small).flatMap(
        lambda kv: [(dst, BETA * kv[1][1] / len(kv[1][0])) for dst in kv[1][0]]
    )
    ranks_small = (
        contribs.reduceByKey(lambda a, b: a + b)
                .mapValues(lambda v: (1.0 - BETA) / N_small + v)
    )

results_small = sorted(ranks_small.collect(), key=lambda x: -x[1])
print(f"Top score: {results_small[0][1]:.4f}  (expected ≈ 0.036)")
print("\nTop 5 nodes:")
for node, score in results_small[:5]:
    print(f"  Node {node:4d}  Score = {score:.6f}")
print("\nBottom 5 nodes:")
for node, score in results_small[-5:]:
    print(f"  Node {node:4d}  Score = {score:.6f}")


# ── Run on whole.txt ─────────────────────────────────────────
print("\n" + "=" * 50)
print("FULL GRAPH (1000 nodes, ~8192 edges)")
print("=" * 50)

N = 1000
raw = sc.textFile("/content/whole.txt")
edges = (
    raw
    .map(lambda line: tuple(map(int, line.strip().split())))
    .filter(lambda e: e[0] != e[1])
    .distinct()
)

links = edges.groupByKey().mapValues(list).cache()
ranks = links.mapValues(lambda _: 1.0 / N)

t_start = time.time()
for i in range(ITERS):
    contribs = links.join(ranks).flatMap(
        lambda kv: [(dst, BETA * kv[1][1] / len(kv[1][0])) for dst in kv[1][0]]
    )
    ranks = (
        contribs.reduceByKey(lambda a, b: a + b)
                .mapValues(lambda v: (1.0 - BETA) / N + v)
    )

print(f"Computed in {time.time() - t_start:.2f}s  ({ITERS} iters, β={BETA})")

results = sorted(ranks.collect(), key=lambda x: -x[1])

print("\nTop 5 nodes (highest PageRank):")
for node, score in results[:5]:
    print(f"  Node {node:4d}  Score = {score:.6f}")

print("\nBottom 5 nodes (lowest PageRank):")
for node, score in results[-5:]:
    print(f"  Node {node:4d}  Score = {score:.6f}")

sc.stop()
print("\nDone ✓")

SMALL GRAPH (53 nodes) — expected top score ≈ 0.036
Nodes: 100
Top score: 0.0357  (expected ≈ 0.036)

Top 5 nodes:
  Node   53  Score = 0.035731
  Node   14  Score = 0.034171
  Node   40  Score = 0.033630
  Node    1  Score = 0.030006
  Node   27  Score = 0.029720

Bottom 5 nodes:
  Node   89  Score = 0.003922
  Node   37  Score = 0.003808
  Node   81  Score = 0.003695
  Node   59  Score = 0.003670
  Node   85  Score = 0.003410

FULL GRAPH (1000 nodes, ~8192 edges)
Computed in 1.86s  (40 iters, β=0.8)

Top 5 nodes (highest PageRank):
  Node  263  Score = 0.002020
  Node  537  Score = 0.001943
  Node  965  Score = 0.001925
  Node  243  Score = 0.001853
  Node  285  Score = 0.001827

Bottom 5 nodes (lowest PageRank):
  Node  408  Score = 0.000388
  Node  424  Score = 0.000355
  Node   62  Score = 0.000353
  Node   93  Score = 0.000351
  Node  558  Score = 0.000329

Done ✓


---
## Summary

| Part | Algorithm | Key Result |
|---|---|---|
| 1  Clustering | Farthest-First (k-Center) | O(n·k) time; 2-approx guarantee |
| 1  Clustering | K-Means++ | O(n·k) time; O(log k) approx guarantee |
| 1  Clustering | Coreset approach | kcenter builds k₁ representatives → kmeans++ refines to k |
| 2  Web Search | Inverted Index + TF-IDF | O(1) word→pages lookup via custom hash table |
| 3  PageRank | Iterative power iteration | 40 iters, β=0.8 on 1000-node graph |

**Observations:**
- K-Means++ almost always outperforms random initialization and can approach kcenter quality.
- The coreset approach (Exp 3) is useful when the dataset is too large to run kmeans++ directly; increasing k₁ improves quality at the cost of a larger intermediate set.
- PageRank converges rapidly on sparse graphs; the top-5 nodes will typically be those with many incoming high-quality links.